# Análise Exploratória de Dados (EDA) - Ações da B3
Estudo estatístico e visual dos ativos listados na B3 extraídos via pipeline ETL (`src/etl.py`), avaliando distribuição de retornos diários, volatilidade histórica e correlação setorial.

In [1]:
import pandas as pd
import plotly.express as px

# Carrega os dados processados pelo ETL
df = pd.read_csv('../data/b3_acoes_tratadas.csv', parse_dates=['Data'])

print(f'Total de registros: {len(df)}')
print(f"Período analisado: {df['Data'].min().date()} até {df['Data'].max().date()}")
display(df.head())
display(df.info())

Total de registros: 3976
Período analisado: 2024-10-31 até 2026-10-30


,Data,Ticker,Setor,Abertura,Maxima,Minima,Fechamento,Volume,Retorno_Diario,MM_20,MM_50,Volatilidade_20d
0,2024-10-31,PETR4,Petróleo e Gás,28.06,28.20,27.87,28.09,25597000,0.17,NaN,NaN,NaN
1,2024-11-01,PETR4,Petróleo e Gás,28.31,28.45,27.61,27.70,30605300,-1.36,NaN,NaN,NaN
2,2024-11-04,PETR4,Petróleo e Gás,28.08,28.12,27.67,27.77,24003600,0.23,NaN,NaN,NaN
3,2024-11-05,PETR4,Petróleo e Gás,27.77,27.84,27.40,27.68,23990600,-0.31,NaN,NaN,NaN
4,2024-11-06,PETR4,Petróleo e Gás,27.41,27.73,27.41,27.69,28440800,0.03,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 3976 entries, 0 to 3975
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Data              3976 non-null   datetime64[us]
 1   Ticker            3976 non-null   str           
 2   Setor             3976 non-null   str           
 3   Abertura          3976 non-null   float64       
 4   Maxima            3976 non-null   float64       
 5   Minima            3976 non-null   float64       
 6   Fechamento        3976 non-null   float64       
 7   Volume            3976 non-null   int64         
 8   Retorno_Diario    3976 non-null   float64       
 9   MM_20             3832 non-null   float64       
 10  MM_50             3592 non-null   float64       
 11  Volatilidade_20d  3824 non-null   float64       
dtypes: datetime64[us](1), float64(8), int64(1), str(2)
memory usage: 445.7 KB


None

In [2]:
# Resumo estatístico: Retorno Médio, Volatilidade e Preço Máximo/Mínimo por Ticker
resumo_ativos = (
    df.groupby(['Ticker', 'Setor'])
    .agg(
        Preco_Medio=('Fechamento', 'mean'),
        Preco_Min=('Minima', 'min'),
        Preco_Max=('Maxima', 'max'),
        Retorno_Medio_Diario=('Retorno_Diario', 'mean'),
        Volatilidade_Media=('Volatilidade_20d', 'mean'),
        Volume_Medio=('Volume', 'mean'),
    )
    .round(2)
    .reset_index()
)

display(resumo_ativos.sort_values(by='Retorno_Medio_Diario', ascending=False))

,Ticker,Setor,Preco_Medio,Preco_Min,Preco_Max,Retorno_Medio_Diario,Volatilidade_Media,Volume_Medio
4,PETR4,Petróleo e Gás,34.46,25.66,59.79,0.14,1.56,40114481.89
3,ITUB4,Financeiro,35.61,22.70,50.85,0.09,1.48,26733563.56
0,ABEV3,Consumo Não Cíclico,13.43,9.90,17.00,0.06,1.45,30965035.61
2,BBDC4,Financeiro,15.37,9.52,22.62,0.04,1.76,37684744.30
6,VALE3,Mineração,61.99,43.55,89.12,0.04,1.53,23290896.38
7,WEGE3,Bens Industriais,45.12,33.70,56.27,0.01,1.82,8443400.20
5,RENT3,Consumo Cíclico,38.05,24.44,54.87,-0.01,2.60,10095374.65
1,BBAS3,Financeiro,22.44,16.13,28.67,-0.05,1.81,29301943.86


In [3]:
# Pivot table para analisar correlação entre os retornos dos ativos
df_pivot = df.pivot(index='Data', columns='Ticker', values='Retorno_Diario')
correlacao = df_pivot.corr().round(2)

fig_corr = px.imshow(
    correlacao,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    title='Matriz de Correlação dos Retornos Diários (B3)',
)
fig_corr.show()

# Gráfico de dispersão Risco (Volatilidade) x Retorno
fig_risco = px.scatter(
    resumo_ativos,
    x='Volatilidade_Media',
    y='Retorno_Medio_Diario',
    text='Ticker',
    color='Setor',
    size='Volume_Medio',
    title='Relação Risco (Volatilidade 20d) vs. Retorno Médio Diário',
)
fig_risco.update_traces(textposition='top center')
fig_risco.show()